# !! Purely for testing, might be outdated

In [1]:
from lerobot.datasets.lerobot_dataset import LeRobotDataset

dataset = LeRobotDataset("ViaCatalyst/abc130k-screwdriver-mcap-lerobot-v3", revision="main")

print(f"Total episodes: {dataset.meta.total_episodes}")
print(f"Total frames: {dataset.meta.total_frames}")
print(f"FPS: {dataset.meta.fps}")

print()

print(len(dataset))

sample = dataset[0]
print("Sample keys:             ", list(sample.keys()))
print("observation.state shape: ", sample["observation.state"].shape)
print("action shape:            ", sample["action"].shape)

print()

for (key, value) in sample.items():
    print(key, ": ", value)


/Users/julien/Developer/Projects/yam-robotics/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total episodes: 1
Total frames: 136
FPS: 20

136
Sample keys:              ['action', 'episode_index', 'frame_index', 'index', 'observation.state', 'task_index', 'timestamp', 'task']
observation.state shape:  torch.Size([26])
action shape:             torch.Size([12])

action :  tensor([ 1.4258e+00,  3.1347e+00,  3.8070e-01,  9.6255e-04, -3.0392e+00,
         1.0563e-01, -8.1300e-02,  1.6103e+00,  2.5314e-01,  1.2095e+00,
        -1.8863e-01, -2.6878e-02])
episode_index :  tensor(0)
frame_index :  tensor(0)
index :  tensor(0)
observation.state :  tensor([ 1.3380e+00,  3.1042e+00,  3.7108e-01,  1.1414e-03, -3.0627e+00,
         8.1344e-02, -8.7930e-02,  1.6150e+00,  2.2908e-01,  1.1484e+00,
        -1.9970e-01, -3.9101e-02,  4.5812e-01, -5.7368e+00,  6.8034e+00,
         2.6154e+00,  7.0818e-02,  1.0501e-01,  7.3260e-03, -2.4420e-03,
        -2.4420e-03, -2.4420e-03, -7.3260e-03, -7.3260e-03, -7.3260e-03,
         9.1575e-03])
task_index :  tensor(0)
timestamp :  tensor(0.)
task :  Put 

In [2]:
delta_t = {
    "observation.state": [-0.05, 0.0],
    "action": [i * 0.05 for i in range(16)],
}

dataset_chunked = LeRobotDataset(
    "ViaCatalyst/abc130k-screwdriver-mcap-lerobot-v3",
    revision="main",
    delta_timestamps=delta_t,
)

sample = dataset_chunked[0]

print(sample["observation.state"].shape)
print(sample["action"].shape)

torch.Size([2, 26])
torch.Size([16, 12])


In [3]:
import torch
from torch.utils.data import DataLoader
import lovely_tensors as lt
from pprint import pprint as pp

lt.monkey_patch()

torch.set_printoptions(
    edgeitems=1,
    threshold=6,
    precision=3,
    linewidth=120,
    sci_mode=False,
)

dataloader = DataLoader(
    dataset_chunked,
    batch_size = 8,
    shuffle=True,
    num_workers=0,
)

batch = next(iter(dataloader))

print("Batch observation.state shape:", batch["observation.state"].shape) # shape = [Batch, Timestamps, Dimensions]
print("Batch action shape:", batch["action"].shape) # shape = [Batch, Timestamps, Dimensions]

# for key, value in batch.items():
#     print(key)
#     print(value)
#     print()

# pp(batch)

Batch observation.state shape: torch.Size([8, 2, 26])
Batch action shape: torch.Size([8, 16, 12])


In [4]:
from pathlib import Path

In [5]:
features = {
    "observation.images.top_camera": {
        "dtype": "video",
        "shape": (480, 640, 3),
        "names": ["height", "width", "channels"],
    },
    # "observation.images.left_camera": {
    #     "dtype": "video",
    #     "shape": (480, 640, 3),
    #     "names": ["height", "width", "channels"],
    # },
    # "observation.images.right_camera": {
    #     "dtype": "video",
    #     "shape": (480, 640, 3),
    #     "names": ["height", "width", "channels"],
    # },
    "observation.state": {
        "dtype": "float32",
        "shape": (14,),
        "names": None,
    },
    "action": {
        "dtype": "float32",
        "shape": (14,),
        "names": None,
    },
}

In [6]:
root = Path("./data/yam_teleop")
mcap_paths = sorted(Path("../../recordings/episodes").glob("*.mcap"))
target_size = (480, 640)

In [7]:
dataset = LeRobotDataset.create(
    repo_id="local/yam_teleop",
    fps=30,
    features=features,
    root=root,
    robot_type="yam_bimanual",
    use_videos=True,
)

In [8]:
print(mcap_paths)

[Path('../../recordings/episodes/1.mcap'), Path('../../recordings/episodes/3.mcap'), Path('../../recordings/episodes/5.mcap'), Path('../../recordings/episodes/8.mcap')]


In [9]:

import struct
import cv2
import numpy as np
from mcap.reader import make_reader

def read_mcap_episode(mcap_path:Path, target_size=(480, 640)): # TODO actual target size?
    frames = []

    with open(mcap_path, "rb") as f:
        reader = make_reader(f)

        current_frame = {}
        for schema, channel, msg in reader.iter_messages():
            topic = channel.topic

            if topic == "/top-camera":
                fmt_len = struct.unpack('<I', msg.data[4:8])[0]
                offset = (8 + fmt_len + 3) & ~3
                data_len = struct.unpack('<I', msg.data[(offset):(offset+4)])[0]
                img_bytes = msg.data[(offset+4):(offset+4+data_len)]

                img_bgr = cv2.imdecode(np.frombuffer(img_bytes, np.uint8), cv2.IMREAD_COLOR)
                img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

                if target_size:
                    img_rgb = cv2.resize(img_rgb, (target_size[1], target_size[0])) # OpenCV uses W,H instead of H,W
                
                current_frame["top_camera"] = img_rgb
            
            elif topic in [
                "/left-arm-state", "/left-ee-state", "/right-arm-state", "/right-ee-state",
                "/left-arm-action", "/left-ee-action", "/right-arm-action", "/right-ee-action",
            ]:
                seq_len = struct.unpack('<I', msg.data[4:8])[0]
                vals = struct.unpack(f'<{seq_len}d', msg.data[12:12+seq_len*8])
                current_frame[topic] = np.array(vals, dtype=np.float32)

            required_topics = [
                "top_camera",
                "/left-arm-state", "/left-ee-state", "/right-arm-state", "/right-ee-state",
                "/left-arm-action", "/left-ee-action", "/right-arm-action", "/right-ee-action",
            ]

            if all(k in current_frame for k in required_topics):
                state_vec = np.concatenate([
                    current_frame["/left-arm-state"],
                    current_frame["/left-ee-state"],
                    current_frame["/right-arm-state"],
                    current_frame["/right-ee-state"],
                ])

                action_vec = np.concatenate([
                    current_frame["/left-arm-action"],
                    current_frame["/left-ee-action"],
                    current_frame["/right-arm-action"],
                    current_frame["/right-ee-action"],
                ])

                frames.append({
                    "observation.images.top_camera": current_frame["top_camera"],
                    "observation.state": state_vec,
                    "action": action_vec,
                    "task": "Bimanual manipulation teleop"
                })
                
                current_frame = {}
    return frames

In [10]:
for mcap_path in mcap_paths:
    episode_frames = read_mcap_episode(mcap_path, target_size=target_size)

    if not episode_frames:
        continue

    for frame in episode_frames:
        dataset.add_frame(frame)
    
    dataset.save_episode()

dataset.finalize()

Map: 100%|██████████| 154/154 [00:00<00:00, 6325.51 examples/s]
Svt[info]: -------------------------------------------
Svt[info]: SVT [version]:	SVT-AV1 Encoder Lib v3.0.0
Svt[info]: SVT [build]  :	Apple LLVM 15.0.0 (clang-1500.3.9.4)	 64 bit
Svt[info]: LIB Build date: Jul  3 2025 03:06:26
Svt[info]: -------------------------------------------
Svt[warn]: Preset M12 is mapped to M10.
Svt[info]: Level of Parallelism: 4
Svt[info]: Number of PPCS 59
Svt[info]: [asm level on system : up to neon_i8mm]
Svt[info]: [asm level selected : up to neon_i8mm]
Svt[info]: -------------------------------------------
Svt[info]: SVT [config]: main profile	tier (auto)	level (auto)
Svt[info]: SVT [config]: width / height / fps numerator / fps denominator 		: 640 / 480 / 30 / 1
Svt[info]: SVT [config]: bit-depth / color format 					: 8 / YUV420
Svt[info]: SVT [config]: preset / tune / pred struct 					: 10 / PSNR / random access
Svt[info]: SVT [config]: gop size / mini-gop size / key-frame type 			: 2 / 16 /

In [11]:
ds = LeRobotDataset(repo_id="data/yam_teleop", root=root)
sample = ds[0]
print(sample["observation.state"].shape)

objc[22651]: Class AVFFrameReceiver is implemented in both /Users/julien/Developer/Projects/yam-robotics/.venv/lib/python3.14/site-packages/av/.dylibs/libavdevice.61.3.100.dylib (0x119dac3a8) and /opt/homebrew/Cellar/ffmpeg/8.1.2/lib/libavdevice.62.3.102.dylib (0x12b960328). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.
objc[22651]: Class AVFAudioReceiver is implemented in both /Users/julien/Developer/Projects/yam-robotics/.venv/lib/python3.14/site-packages/av/.dylibs/libavdevice.61.3.100.dylib (0x119dac3f8) and /opt/homebrew/Cellar/ffmpeg/8.1.2/lib/libavdevice.62.3.102.dylib (0x12b960378). This may cause spurious casting failures and mysterious crashes. One of the duplicates must be removed or renamed.


torch.Size([14])
